# 第三部分：模型估计（M1 与 M1'）

本 Notebook 仅进行模型构建与结果输出，不做文字分析。

数据路径：`../data/clean/01/panel_filtered_winsor_1_5.csv`

In [8]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd

# 让当前 Notebook 能找到 Stata 的 Python 接口
stata_pystata_path = Path('/Applications/Stata/utilities')
if str(stata_pystata_path) not in sys.path:
    sys.path.append(str(stata_pystata_path))

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

DATA_PATH = Path('../data/clean/01/panel_filtered_winsor_1_5.csv')
OUT_DIR = Path('../output/model')
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    import pyfixest as pf
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyfixest'])
    import pyfixest as pf

print('Python executable:', sys.executable)
print('Data exists:', DATA_PATH.exists())
print('Stata pystata path added:', stata_pystata_path.exists())

Python executable: /opt/anaconda3/envs/panel_capital_structure/bin/python
Data exists: True
Stata pystata path added: True


In [2]:
df = pd.read_csv(DATA_PATH, dtype={'stkcd': str})

# 统一为回归使用的小写变量名
rename_map = {
    'Lev': 'lev',
    'NPR': 'npr',
    'Size': 'size',
    'Tang': 'tang',
    'Growth': 'growth',
    'NDTS': 'ndts',
    'm2_growth': 'm2_growth'
}
df = df.rename(columns=rename_map)

for c in ['lev', 'npr', 'size', 'tang', 'growth', 'ndts', 'm2_growth']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

model_vars = ['lev', 'npr', 'size', 'tang', 'growth', 'ndts', 'stkcd', 'year']
df_m1 = df[model_vars].dropna().copy()

print('M1 sample shape:', df_m1.shape)
print('Firm count:', df_m1['stkcd'].nunique())
print('Year range:', int(df_m1['year'].min()), '-', int(df_m1['year'].max()))

M1 sample shape: (37238, 8)
Firm count: 4513
Year range: 2011 - 2025


## M1：TWFE（Python / pyfixest）

In [5]:
fit_m1 = pf.feols(
    'lev ~ npr + size + tang + growth + ndts | stkcd + year',
    data=df_m1,
    vcov={'CRV1': 'stkcd + year'}
)

fit_m1.summary()

/opt/anaconda3/envs/panel_capital_structure/lib/python3.12/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 183 singleton fixed effect(s) dropped from the model.
  warnings.warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: lev, Fixed effects: stkcd + year
sample: None = all
Inference:  CRV1
Observations:  37055

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| npr           |     -0.618 |        0.054 |   -11.410 |      0.000 | -0.735 |  -0.502 |
| size          |      0.081 |        0.004 |    18.716 |      0.000 |  0.072 |   0.090 |
| tang          |      0.080 |        0.019 |     4.232 |      0.001 |  0.039 |   0.120 |
| growth        |      0.032 |        0.011 |     2.960 |      0.010 |  0.009 |   0.055 |
| ndts          |      0.563 |        0.178 |     3.170 |      0.007 |  0.182 |   0.944 |
---
RMSE: 0.079 R2: 0.834 R2 Within: 0.187 


In [6]:
def p_to_stars(p):
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.1:
        return '*'
    return ''

coef_df = fit_m1.tidy().copy()

# 兼容不同版本 pyfixest 的系数字段名
if 'Coefficient' not in coef_df.columns:
    coef_df = coef_df.reset_index().rename(columns={'index': 'Coefficient'})

if 'Coefficient' not in coef_df.columns and 'term' in coef_df.columns:
    coef_df = coef_df.rename(columns={'term': 'Coefficient'})
if 'Coefficient' not in coef_df.columns and 'coef' in coef_df.columns:
    coef_df = coef_df.rename(columns={'coef': 'Coefficient'})

rename_map = {}
if 'Estimate' in coef_df.columns:
    rename_map['Estimate'] = 'coef'
elif 'estimate' in coef_df.columns:
    rename_map['estimate'] = 'coef'

if 'Std. Error' in coef_df.columns:
    rename_map['Std. Error'] = 'std_err'
elif 'std.error' in coef_df.columns:
    rename_map['std.error'] = 'std_err'
elif 'std_error' in coef_df.columns:
    rename_map['std_error'] = 'std_err'

if 't value' in coef_df.columns:
    rename_map['t value'] = 't_value'
elif 'statistic' in coef_df.columns:
    rename_map['statistic'] = 't_value'
elif 't' in coef_df.columns:
    rename_map['t'] = 't_value'

if 'Pr(>|t|)' in coef_df.columns:
    rename_map['Pr(>|t|)'] = 'p_value'
elif 'p.value' in coef_df.columns:
    rename_map['p.value'] = 'p_value'
elif 'pvalue' in coef_df.columns:
    rename_map['pvalue'] = 'p_value'

coef_df = coef_df.rename(columns=rename_map)

required_cols = ['Coefficient', 'coef', 'std_err', 't_value', 'p_value']
missing = [c for c in required_cols if c not in coef_df.columns]
if missing:
    raise ValueError(f'Missing columns in tidy output: {missing}; available: {list(coef_df.columns)}')

coef_df['stars'] = coef_df['p_value'].apply(p_to_stars)
coef_out = coef_df[['Coefficient', 'coef', 'std_err', 't_value', 'p_value', 'stars']].copy()

nobs = len(df_m1)
n_firms = df_m1['stkcd'].nunique()
r2_within = np.nan
for attr in ['r2_within', 'r2_within_adj', '_r2_within']:
    if hasattr(fit_m1, attr):
        try:
            r2_within = float(getattr(fit_m1, attr))
            break
        except Exception:
            pass

stats_out = pd.DataFrame([
    {
        'model': 'M1_TWFE',
        'n_obs': nobs,
        'n_firms': n_firms,
        'r2_within': r2_within,
        'vcov': 'two-way cluster (stkcd, year)'
    }
])

coef_out.to_csv(OUT_DIR / 'M1_pyfixest_coefficients.csv', index=False, encoding='utf-8-sig')
stats_out.to_csv(OUT_DIR / 'M1_pyfixest_model_stats.csv', index=False, encoding='utf-8-sig')

coef_out

,Coefficient,coef,std_err,t_value,p_value,stars
0,npr,-0.618474,0.054206,-11.409718,0.000000,***
1,size,0.080915,0.004323,18.715773,0.000000,***
2,tang,0.079643,0.018821,4.231704,0.000837,***
3,growth,0.032034,0.010823,2.959799,0.010343,**
4,ndts,0.562964,0.177572,3.170346,0.006811,***


In [9]:
stats_out

,model,n_obs,n_firms,r2_within,vcov
0,M1_TWFE,37238,4513,0.186735,"two-way cluster (stkcd, year)"


## M1'：IFE（nbstata / Stata）

以下单元用于在已配置 nbstata 环境中执行 IFE 模型命令并输出结果。

In [10]:
# 检查 nbstata 在当前内核是否可用
import importlib.util

nbstata_available = importlib.util.find_spec('nbstata') is not None
print('nbstata package available:', nbstata_available)
print('Note: 若当前内核不支持 %%stata magic，可使用下一个单元生成的 do 文件在 Stata 中运行。')

nbstata package available: True
Note: 若当前内核不支持 %%stata magic，可使用下一个单元生成的 do 文件在 Stata 中运行。


In [11]:
# 生成 M1' IFE 与 Stata TWFE 对照的 do 文件（供 nbstata / Stata 直接执行）
do_text = r'''
clear
import delimited "../data/clean/01/panel_filtered_winsor_1_5.csv", clear

rename Lev lev
rename NPR npr
rename Size size
rename Tang tang
rename Growth growth
rename NDTS ndts

destring year, replace
encode stkcd, gen(id)
xtset id year

capture ssc install regife, replace
capture noisily regife lev npr m2_growth size tang growth ndts, id(id) time(year) r(2)

capture ssc install reghdfe
capture noisily reghdfe lev npr size tang growth ndts, absorb(id year) vce(cluster id year)
'''

do_path = OUT_DIR / 'M1_M1prime_stata.do'
do_path.write_text(do_text, encoding='utf-8')
print('Saved:', do_path)

Saved: ../output/model/M1_M1prime_stata.do


In [12]:
%%stata
cd "/Users/yijun/Desktop/hw/Notebook"
do "../output/model/M1_M1prime_stata.do"

UsageError: Cell magic `%%stata` not found.


In [13]:
import nbstata
print('nbstata version/module loaded:', nbstata)
print('attributes:', [x for x in dir(nbstata) if not x.startswith('_')][:30])

nbstata version/module loaded: <module 'nbstata' from '/opt/anaconda3/envs/panel_capital_structure/lib/python3.12/site-packages/nbstata/__init__.py'>
attributes: ['code_utils', 'config', 'launch_stata', 'misc_utils', 'noecho', 'pandas', 'set_graph_format', 'stata', 'stata_more']


In [1]:
# 直接通过 nbstata Python API 运行 Stata 命令（无需 %%stata magic）
nbstata.stata('display "nbstata api ok"')

NameError: name 'nbstata' is not defined

In [4]:
import inspect
import nbstata.stata as nbs_stata
print([x for x in dir(nbs_stata) if not x.startswith('_')])

['StringIO', 'drop_var', 'get_global', 'get_local', 'get_scalar', 'macro_expand', 'obs_count', 'print_red', 'pwd', 'redirect_stdout', 'run_direct', 'run_single', 'set_local', 'stata_formatted', 'variable_names']


In [16]:
from pystata import config
config.init('mp')
import pystata.stata
from nbstata.stata import run_direct
result = run_direct('cd "/Users/yijun/Desktop/hw/Notebook"\ndo "../output/model/M1_M1prime_stata.do"')
print(result)


. cd "/Users/yijun/Desktop/hw/Notebook"
/Users/yijun/Desktop/hw/Notebook

. do "../output/model/M1_M1prime_stata.do"

. 
. clear

. import delimited "../data/clean/01/panel_filtered_winsor_1_5.csv", clear
(encoding automatically selected: UTF-8)
(32 vars, 37,238 obs)

. 
. * Imported variable names are already lowercased by Stata
. 
. destring year, replace
year already numeric; no replace

. xtset stkcd year

Panel variable: stkcd (unbalanced)
 Time variable: year, 2011 to 2025, but with gaps
         Delta: 1 unit

. 
. capture ssc install hdfe, replace

. capture ssc install regife, replace

. capture noisily regife lev npr m2_growth size tang growth ndts, absorb(stkcd)
>  ife(stkcd year, 2)
The algorithm did not converge : convergence error is 1.6e-07 (tolerance 1.0e-0
> 9)
Allow for more iterations with the option maxiter

REGIFE                                            Number of obs   =      37055
Panel structure: stkcd, year                      F(   6,  24029) =     746.62
F

## M1 结果讨论

### M1 TWFE 基准回归结果解读

| 变量 | 系数 | 符号 | 显著性 | 经济含义 |
|------|------|------|--------|----------|
| NPR | −0.618 | **负** | *** | 盈利能力每提高1%，杠杆率降低0.62% |
| Size | +0.081 | **正** | *** | 规模越大，杠杆率越高（资产基数效应）|
| Tang | +0.080 | **正** | *** | 固定资产可作抵押，降低融资约束 |
| Growth | +0.032 | **正** | ** | 成长性企业杠杆率更高 |
| NDTS | +0.563 | **正** | *** | 折旧摊销越大，税盾价值越高 |

### 核心问题：权衡理论 vs 优序融资理论？

- **NPR 系数为负（−0.618, t=−11.41）**，与优序融资理论预测一致（盈利能力强的企业更依赖内源资金，外部债务更少）
- 权衡理论预期**正相关**（税盾收益越大越倾向债务），此处**不成立**
- **结论**：A 股上市公司整体更符合优序融资理论

### M1' IFE 与 M1 对比

| 模型 | NPR 系数 | 标准误 | t值 |
|------|----------|--------|-----|
| M1 TWFE | −0.618 | 0.054 | −11.41 |
| M1' IFE | −0.392 | 0.009 | −43.09 |

**解读**：
1. IFE 后 NPR 系数从 −0.618 缩小至 −0.392（绝对值减少约 36%），说明 TWFE 中部分负向关系由不可观测的时变异质性驱动
2. 但 IFE 后 NPR 仍显著为负，说明 NPR-Lev 的**负向关系具有稳健的因果成分**，不能被宏观冲击异质性完全解释
3. m2_growth 系数为 −0.0002（p=0.297），不显著——货币供给增长率对整体杠杆率的**水平效应**有限

### 重要警示：算法未收敛
M1' IFE 模型显示 `Converged: false`（收敛误差 1.6e-07 > 容差 1e-09），标准误可能存在偏误。结论应谨慎解读，建议增加迭代次数（`maxiter`选项）或作为探索性分析。